#### 2026-04-22/23 First repeat analysis run

In [ ]:
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/951/802/345/GCA_951802345.1_Asativa_cv_Williams_v1.0/GCA_951802345.1_Asativa_cv_Williams_v1.0_genomic.fna.gz &
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/951/802/345/GCA_951802345.1_Asativa_cv_Williams_v1.0/GCA_951802345.1_Asativa_cv_Williams_v1.0_genomic.gbff.gz &

##### Assembly QC (SeqKit + BUSCO)

In [ ]:
# envs
conda create -n assembly-qc
conda install seqkit -c bioconda
conda install busco -c bioconda

In [ ]:
seqkit stats \
--all \
--tabular \
*.fna.gz > ./stats/seqkit-stats_26-04-23.tsv &

##### Repeat search

In [ ]:
conda activate EDTA
perl ./tools/EDTA/EDTA.pl \
--genome ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz \
--species other \
--step all \
--anno 1 \
--sensitive 1 \
--evaluate 1 \
--threads 20 &

#### 2026-04-23 Data evaluation & fixing the problem above

In [ ]:
zgrep "^>" ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz | sort | uniq -d

In [ ]:
zcat ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz | head

In [ ]:
zcat ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz | tail

In [ ]:
grep -c "A" ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz
grep -c "a" ./data/assemblies/Asativa_cv-Williams_v1.0.fna.gz

In [ ]:
conda activate assembly-qc

seqkit rename \
./data/assemblies/Asativa_cv-Williams_v1.0.fna \
--threads 12 > \
./data/assemblies/renamed_fna/Asativa_cv-Williams_v1.0_wo-duplicates.fna

In [ ]:
tmux attach
perl ./tools/EDTA/EDTA.pl \
--genome ./data/assemblies/renamed_fna/Asativa_cv-Williams_v1.0_wo-duplicates.fna \
--species others \
--step all \
--anno 1 \
--sensitive 1 \
--evaluate 1 \
--threads 20 &

#### 2026-05-28 Second try of the analysis

cd ./data/EDTA/Asativa_cv-Williams_v1.0/
perl ../../../tools/EDTA/EDTA.pl \
--genome ../../../data/assemblies/renamed_fna/Asativa_cv-Williams_v1.0_wo-duplicates.fna \
--species others \
--step all \
--anno 1 \
--sensitive 1 \
--evaluate 1 \
--threads 20 \
> EDTA.Asativa_cv-Williams_v1.0.2026-05-28.log \
&

Let's try find the problem

#### 2026-05-29 Second try of the analysis -- continuation

Seems to be working...

#### 2026-06-01 New tool

Well, the EDTA runned for 2+ days, however it seems like it would have run much longer (if I didn't stopped it due to my concerns about the server limitations) - so I discussed the problem my collegue, and he recommended his tool: https://github.com/soyboy-hub/JumpORF

So we try it

test run was a success! \
let's do a test run with my data!

It didn't worked :c

#### 2026-06-08 Let's start a new try for EDTA

It worked!!! \
Let's do EDTA for every chromosome

#### 2026-06-14 EDTA results visualization

Hi! My EDTA annotation still in proccess (3 chrs need 1 day for annotation, so I daily check the result) -- but I can start preparing for the analysis of results via Circos

Firstly, I wrote (with AI assistance) a code to create a TSV file of the kaaryotype for Circos

Then I've tried the first Circos for my data

#### 2026-06-15 EDTA results visualization (continuation)

For links (which will show the distribution of repeat in the genome) I've downloaded the TE-library, constructed by EDTA

Let's map library of each chr to the genome

#### 2026-06-16 EDTA results visualization (continuation)

Let's create tracks for repeats density per chromosome

In [3]:
chrs = [f"{num}{letter}" for num in range(1, 6) for letter in ['A', 'C', 'D']]
print(chrs)
for chrom in chrs:
    print("python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_" + chrom + ".fasta.mod.EDTA.TEanno.gff3 -o As001_" + chrom)

['1A', '1C', '1D', '2A', '2C', '2D', '3A', '3C', '3D', '4A', '4C', '4D', '5A', '5C', '5D']
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_1A.fasta.mod.EDTA.TEanno.gff3 -o As001_1A
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_1C.fasta.mod.EDTA.TEanno.gff3 -o As001_1C
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_1D.fasta.mod.EDTA.TEanno.gff3 -o As001_1D
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_2A.fasta.mod.EDTA.TEanno.gff3 -o As001_2A
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_2C.fasta.mod.EDTA.TEanno.gff3 -o As001_2C
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_2D.fasta.mod.EDTA.TEanno.gff3 -o As001_2D
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_3A.fasta.mod.EDTA.TEanno.gff3 -o As001_3A
python ../../../../tools/Circos_fromGFF3toDensityTSV.py -i ./chr_3C.fasta.mod.EDTA.TEanno.gff3 -o As001_3C
python ../../../../tools/Circos_fromGFF3toDensityTSV.

In [ ]:
circos -conf general.conf \
-outputdir ./images/ \
-outputfile As001_Chrs_1to5_v1.1 

#### 2026-06-21 TE library creation & exploration

Let's merge our TElibs from every chromosome into one whole A.sativa TE library

In [24]:
chrs = [f"{num}{letter}" for num in range(1, 8) for letter in ['A', 'C', 'D']]
# print(chrs)
for chrom in chrs:
    # print("mv ../../../EDTA/As001/" + chrom + "/chr_" + chrom + ".fasta.mod.EDTA.TEanno.gtf ./")
    # print("mv ../../../../EDTA/As001/" + chrom + "/chr_" + chrom + ".fasta.mod.EDTA.TElib.fa ./") 
    print("python ../../../../../tools/AddPrefixGTF.py ./raw/chr_" + chrom + ".fasta.mod.EDTA.TEanno.gtf ./prefixed/As001." + chrom + ".EDTA.TEanno.prefixed.gtf --prefix " + chrom + "_")
    # print("python ../../../../../tools/AddPrefixFA.py chr_" + chrom + ".fasta.mod.EDTA.TElib.fa As001." + chrom + ".EDTA.TElib.prefixed.fa --prefix " + chrom + "_")

python ../../../../../tools/AddPrefixGTF.py ./raw/chr_1A.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.1A.EDTA.TEanno.prefixed.gtf --prefix 1A_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_1C.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.1C.EDTA.TEanno.prefixed.gtf --prefix 1C_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_1D.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.1D.EDTA.TEanno.prefixed.gtf --prefix 1D_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_2A.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.2A.EDTA.TEanno.prefixed.gtf --prefix 2A_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_2C.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.2C.EDTA.TEanno.prefixed.gtf --prefix 2C_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_2D.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.2D.EDTA.TEanno.prefixed.gtf --prefix 2D_
python ../../../../../tools/AddPrefixGTF.py ./raw/chr_3A.fasta.mod.EDTA.TEanno.gtf ./prefixed/As001.3A.EDTA.TEanno.prefixed.gtf --prefix 3A_
python ../../

In [ ]:
cat ./prefixed/*.prefixed.fa > As001.EDTA.TElib.withdupls.fa
cat ./prefixed/*.prefixed.gtf > As001.EDTA.TElib.withdupls.gtf

In [ ]:
python ../../../tools/DeduplTElib.py --fasta As001.EDTA.TElib.withdupls.fa --gtf As001.EDTA.TElib.withdupls.gtf -o As001.EDTA.TElib.dedupled
# [INFO] Deduplicating FASTA...
# [INFO] FASTA records: 92300
# [INFO] Unique sequences: 92225
# [INFO] Duplicates removed: 75
# [INFO] Remapping GTF...
# [INFO] GTF records processed: 9728639
# [INFO] Records remapped: 536
# [INFO] Done.
# [INFO] FASTA:   As001.EDTA.TElib.dedup.fa
# [INFO] GTF:     As001.EDTA.TElib.remapped.gtf
# [INFO] Mapping: As001.EDTA.TElib.id_mapping.tsv

SyntaxError: invalid syntax (2226248079.py, line 1)

In [ ]:
grep "^>" As001.EDTA.TElib.dedup.fa \
| sed 's/^>//' \
| cut -d'#' -f2 \
| sort | uniq -c > As001.EDTA.TElib.description.txt

#       1 DNA
#     122 DNA/CMC-EnSpm
#      84 DNA/DTA
#    7298 DNA/DTC
#    7558 DNA/DTH
#    5832 DNA/DTM
#   10098 DNA/DTT
#       7 DNA/hAT-Ac
#      43 DNA/hAT-Tip100
#    2432 DNA/Helitron
#     127 DNA/MULE-MuDR
#      25 DNA/PIF-Harbinger
#       2 DNA/TcMar-Stowaway
#       1 LINE
#     972 LINE/L1
#       1 LINE/L1-Tx1
#       3 LINE/R1-LOA
#      19 LINE/RTE-BovB
#       1 LTR
#   11540 LTR/Copia
#   20480 LTR/Gypsy
#       3 LTR/Solo
#      35 LTR/TRIM
#   12730 LTR/unknown
#      99 MITE/DTA
#     454 MITE/DTC
#    3680 MITE/DTH
#    1602 MITE/DTM
#    4600 MITE/DTT
#      17 RC/Helitron
#      15 rRNA
#       2 tRNA
#    2342 unknown

In [ ]:
seqkit stats As001.EDTA.TElib.dedup.fa > As001.EDTA.TElib.stats

# file                       format  type  num_seqs      sum_len  min_len  avg_len  max_len
# As001.EDTA.TElib.dedup.fa  FASTA   DNA     92,225  184,620,080       80  2,001.8   23,259

In [ ]:
#TEsorter 1.5.1
TEsorter \
As001.EDTA.TElib.dedup.fa \
-db rexdb-plant \
-p 28 &

In [ ]:
awk 'NR>1{print $4}' *.cls.tsv \
| sort | uniq -c | sort -nr
#    9657 unknown
#    2837 Retand
#    2016 Ale
#    1361 Tekay
#    1325 Athila
#    1276 CRM
#     822 Angela
#     681 Reina
#     521 Ikeros
#     518 Ivana
#     423 SIRE
#     261 Bianca
#     159 Ogre
#     143 TAR
#     100 Tork
#      86 mixture
#       8 Alesia
#       6 Ferney
#       5 Galadriel
#       3 TatII
#       2 Gymco-II
#       2 chromo-unclass
#       1 Tcn1                                                                                                                                                                      1 Tatius
#       1 Selgy
#       1 Gymco-III
#       1 Gymco-I
#       1 Chlamyvir

In [ ]:
 awk 'NR>1 && $4=="unknown"{print $2"/"$3}' *.cls.tsv \
| sort | uniq -c | sort -nr
#    4481 LTR/Gypsy
#    1637 LTR/Copia
#    1286 TIR/PIF_Harbinger
#     771 TIR/MuDR_Mutator
#     722 LINE/unknown
#     375 TIR/EnSpm_CACTA
#     161 TIR/Tc1_Mariner
#      90 TIR/hAT
#      68 LTR/mixture
#      64 Helitron/unknown
#       2 mixture/mixture

In [ ]:
bash StatsForLineages.sh > As001.EDTA.TElib.Gypsy-Copia.stats
# Gypsy_unknown   4481    1205.0  684.0
# Gypsy_Retand    2837    4477.9  3713.0
# Copia_Ale       2016    4143.1  4582.0
# Copia_unknown   1637    987.5   573.0
# Gypsy_Tekay     1361    2413.6  2036.0
# Gypsy_Athila    1325    2725.6  2331.0
# Gypsy_CRM       1276    2048.0  1749.5
# Copia_Angela    822     1639.4  1233.5
# Gypsy_Reina     681     4369.1  4705.0
# Copia_Ikeros    521     4963.9  5120.0
# Copia_Ivana     518     3668.0  4332.5
# Copia_SIRE      423     6410.8  7069.0
# Copia_Bianca    261     5300.8  6204.0
# Gypsy_Ogre      159     6866.1  7551.0
# Copia_TAR       143     3481.8  4057.0
# Copia_Tork      100     2768.8  2574.0
# Copia_mixture   46      5218.2  4574.5
# Gypsy_mixture   40      3910.9  3730.5
# Copia_Alesia    8       1362.4  1317.5
# Gypsy_Ferney    6       2089.3  1710.0
# Gypsy_Galadriel 5       1698.2  2075.0
# Gypsy_TatII     3       596.0   702.0
# Copia_Gymco-II  2       721.0   721.0
# Gypsy_chromo-unclass    2       1023.5  1023.5
# Copia_Gymco-I   1       168.0   168.0
# Copia_Gymco-III 1       550.0   550.0
# Gypsy_Chlamyvir 1       678.0   678.0
# Gypsy_Selgy     1       878.0   878.0
# Gypsy_Tatius    1       3180.0  3180.0
# Gypsy_Tcn1      1       710.0   710.0
# Group   N       Mean    Median

In [ ]:
grep '^>' As001.EDTA.TElib.dedup.TEsorted.fa \
| sed 's/^>//' \
| awk '
{
    split($1,a,"#")
    split($2,b,"#")

    print a[1]"\t"a[2]"\t"b[2]
}
' OFS="\t" \
> As001.EDTA.TElib.dedup.TEsorted.classes.tsv

In [ ]:
awk '
BEGIN{
    OFS="\t"
    print "ID","TEsorter","EDTA","Final"
}

{
    id=$1
    ts=$2
    edta=$3

    if(ts!="Unknown")
        final=ts
    else if(edta!="unknown" && edta!="Unknown")
        final=edta
    else
        final="unknown"

    print id,ts,edta,final
}
' As001.EDTA.TElib.dedup.TEsorted.classes.tsv > As001.EDTA.TElib.dedup.TEsorted.classes.final.tsv

In [ ]:
awk '
BEGIN{
    FS=OFS="\t"
}

FNR==NR{
    if(NR>1)
        cls[$1]=$4
    next
}

/^>/{

    hdr=substr($0,2)

    split(hdr,a,"#")

    id=a[1]

    print ">"id"#"cls[id]

    next
}

{
    print
}
' As001.EDTA.TElib.dedup.TEsorted.classes.final.tsv As001.EDTA.TElib.dedup.TEsorted.fa \
> As001.EDTA.TElib.dedup.TEsorted.classified.fa

##### Circos

In [ ]:
awk 'BEGIN{OFS="\t"} {print $1, $4}' As001.EDTA.TElib.dedup.TEsorted.classes.final.tsv > As001.EDTA.TElib.dedup.TEsorted.classes.final-short.tsv